# Customer Support Ticket Router

Run these cells from top to bottom in Google Colab using a standard Python runtime. No LLM or API key is required.

The state holds the ticket, category, and reply. The classify node checks billing keywords; matching tickets go to billing, and all others go to technical support. Billing takes priority when a ticket mentions both topics.


In [ ]:
%pip install -q langgraph


In [ ]:
import re
from typing import Literal, TypedDict

from IPython.display import Image, display
from langgraph.graph import END, START, StateGraph


## Define the state and nodes


In [ ]:
class TicketState(TypedDict):
    ticket: str
    category: Literal["billing", "technical"] | None
    reply: str


BILLING_KEYWORDS = {"billing", "invoice", "payment", "refund", "charge", "charged"}


def classify(state: TicketState) -> dict:
    words = set(re.findall(r"[a-z]+", state["ticket"].lower()))
    category = "billing" if words & BILLING_KEYWORDS else "technical"
    return {"category": category}


def billing(state: TicketState) -> dict:
    return {"reply": "Our billing team can help. Please provide your invoice or order number."}


def technical(state: TicketState) -> dict:
    return {"reply": "Our technical team can help. Please share the error message and steps to reproduce it."}


def route_ticket(state: TicketState) -> Literal["billing", "technical"]:
    if state["category"] == "billing":
        return "billing"
    return "technical"


## Build the graph

A conditional edge selects the support node. Each support node writes a reply and ends.


In [ ]:
builder = StateGraph(TicketState)
builder.add_node("classify", classify)
builder.add_node("billing", billing)
builder.add_node("technical", technical)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route_ticket,
    {"billing": "billing", "technical": "technical"},
)
builder.add_edge("billing", END)
builder.add_edge("technical", END)

graph = builder.compile()


## Draw the graph

This cell uses an online Mermaid rendering service. The diagram contains graph structure only, without ticket text.


In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))


## Run one sample ticket for each path


In [ ]:
sample_tickets = [
    "I was charged twice for my subscription. Can I get a refund?",
    "The app crashes whenever I try to log in.",
]

for ticket in sample_tickets:
    result = graph.invoke({"ticket": ticket, "category": None, "reply": ""})
    print(f"Ticket: {result['ticket']}")
    print(f"Category: {result['category']}")
    print(f"Reply: {result['reply']}")
    print()
